# Simulador 17.1 — Curvas de Indiferencia: Dos Arquitecturas de Preferencia
### *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Capítulo 17: Optimización en Equilibrio**

---

Este simulador compara las dos familias de funciones de utilidad que aparecen en los modelos de optimización del comportamiento:

| | Función de potencia | Función cuadrática |
|---|---|---|
| Forma | $U = R^{s_1} + r^{s_2}$ | $U = -a(R-R_0)^2 - b(r-r_0)^2$ |
| Curvas de indiferencia | Hipérbolas convexas | Elipses centradas en $B_0$ |
| Propiedad clave | Siempre vale más (ganancias decrecientes) | Tiene punto de saturación $B_0$ |
| Genera función bitónica en RV | No (con 2 comportamientos) | Sí |

**Origen psicofísico de la convexidad**: la función de potencia es de ganancias decrecientes — la misma propiedad de la función psicofísica de Stevens. Cuando ya se tiene mucho de un comportamiento, sacrificar un poco de él requiere poca compensación del otro. Esa dependencia del nivel produce curvas de indiferencia convexas.

---


## Guía de exploración

**Panel izquierdo (Función de potencia)**
- Reduce $s_1$ y $s_2$ hacia 0. ¿Qué le ocurre a las curvas? ¿Qué implica para la sustituibilidad entre trabajo y consumo?
- Aumenta $s$ a valores mayores de 1. ¿Las curvas se vuelven convexas o cóncavas? ¿Qué interpretación tiene un exponente mayor que 1?

**Panel derecho (Función cuadrática / Bliss point)**
- Mueve el bliss point hacia $R_0 = 1$, $r_0 = 12$. ¿Qué dice eso sobre un organismo que prefiere casi no trabajar y consumir mucho?
- Aumenta el cociente $b/a$ de 1.0 a 4.0. ¿Cómo cambia la forma de las elipses? ¿Qué implica un $b/a$ grande?

**Comparación**
- ¿En qué situación biológica sería más apropiada la función de potencia que la cuadrática? ¿Y al revés?
- ¿Por qué la función cuadrática produce una función de respuesta bitónica bajo RV, mientras que la de potencia no puede hacerlo (con solo dos comportamientos)?


In [ ]:
# ── Colab: ejecuta esta celda primero ─────────────────────────────────────────
# Si ves un error de widgets, descomenta las dos líneas siguientes:
# !pip install -q ipywidgets
# from google.colab import output; output.enable_custom_widget_manager()

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout
import warnings
warnings.filterwarnings('ignore')

AZUL    = '#2C5282'
NARANJA = '#C05621'
VERDE   = '#276749'
GRIS    = '#718096'

try:
    matplotlib.rc('font', family='Georgia')
except Exception:
    pass


def sim1(s1=0.50, s2=0.50, R0=8.0, r0=6.0, b_sobre_a=1.5):
    LMAX = 20
    R = np.linspace(0.05, LMAX, 400)
    r = np.linspace(0.05, LMAX, 400)
    RR, rr = np.meshgrid(R, r)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
    fig.patch.set_facecolor('white')

    # ─── Panel izquierdo: funciones de potencia ────────────────────────────
    ax = axes[0]
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

    U = RR**s1 + rr**s2
    Umax = np.nanmax(U)
    levels = np.linspace(Umax * 0.12, Umax * 0.88, 8)
    cs = ax.contour(RR, rr, U, levels=levels,
                    colors=[AZUL] * len(levels), linewidths=1.7, alpha=0.85)
    ax.clabel(cs, fmt='%.1f', fontsize=8, colors=GRIS)

    ax.annotate('', xy=(16.5, 16.5), xytext=(11, 11),
                arrowprops=dict(arrowstyle='->', color=VERDE, lw=2.2,
                                mutation_scale=22))
    ax.text(16.7, 15.5, 'mayor\nutilidad', color=VERDE, fontsize=9)

    if max(s1, s2) < 0.35:
        msg = 's pequeño: muy curvadas\nbaja sustituibilidad'
    elif min(s1, s2) > 0.75:
        msg = 's grande: más lineales\nalta sustituibilidad'
    else:
        msg = 'Sustitutos imperfectos\n(caso biológico típico)'
    ax.text(0.50, 0.06, msg, transform=ax.transAxes, fontsize=9,
            color=GRIS, ha='center',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor=GRIS, alpha=0.9))

    title1 = ('Función de potencia\n'
               r'$U = R^{s_1} + r^{s_2}$' + f'    $s_1={s1:.2f},\\ s_2={s2:.2f}$')
    ax.set_xlim(0, LMAX); ax.set_ylim(0, LMAX)
    ax.set_xlabel('Tasa de trabajo  $R$  (resp/min)', fontsize=11)
    ax.set_ylabel('Tasa de consumo  $r$  (ref/min)', fontsize=11)
    ax.set_title(title1, fontsize=12, color=AZUL)
    ax.grid(True, alpha=0.20, color=GRIS)

    # ─── Panel derecho: función cuadrática / bliss point ──────────────────
    ax = axes[1]
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

    a_w, b_w = 1.0, b_sobre_a
    U_q = -(a_w * (RR - R0)**2 + b_w * (rr - r0)**2)
    U_min = np.nanmin(U_q)
    levels_q = np.linspace(U_min * 0.85, U_min * 0.04, 8)

    ax.contour(RR, rr, U_q, levels=levels_q,
               colors=[AZUL] * 8, linewidths=1.7, alpha=0.85)

    bp_label = f'Bliss point $B_0=({R0:.0f},{r0:.0f})$'
    ax.plot(R0, r0, 'o', color=NARANJA, markersize=12, zorder=6,
            label=bp_label)
    ann_x = min(R0 + 2.5, LMAX - 1.5)
    ann_y = min(r0 + 2.0, LMAX - 1.5)
    ax.annotate(f'$B_0=({R0:.0f},{r0:.0f})$',
                xy=(R0, r0), xytext=(ann_x, ann_y),
                fontsize=10, color=NARANJA, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=NARANJA, lw=1.5))

    if R0 < LMAX - 6:
        ax.annotate('', xy=(R0 + 5.5, r0 + 0.5), xytext=(R0 + 0.8, r0 + 0.1),
                    arrowprops=dict(arrowstyle='->', color=GRIS, lw=1.5))
        ax.text(R0 + 5.7, r0 - 0.3, 'menor\nutilidad', color=GRIS, fontsize=9)

    ax.text(0.50, 0.06,
            'Elipses centradas en $B_0$\nMás allá del bliss point, más ≠ mejor',
            transform=ax.transAxes, fontsize=9, color=GRIS, ha='center',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor=GRIS, alpha=0.9))

    title2 = ('Función cuadrática  (bliss point)\n'
               r'$U=-a(R-R_0)^2 - b(r-r_0)^2$' + f'    $b/a={b_sobre_a:.1f}$')
    ax.set_xlim(0, LMAX); ax.set_ylim(0, LMAX)
    ax.set_xlabel('Tasa de trabajo  $R$  (resp/min)', fontsize=11)
    ax.set_ylabel('Tasa de consumo  $r$  (ref/min)', fontsize=11)
    ax.set_title(title2, fontsize=12, color=AZUL)
    ax.grid(True, alpha=0.20, color=GRIS)
    ax.legend(fontsize=10, loc='upper right')

    fig.suptitle('Curvas de Indiferencia: Dos Arquitecturas de Preferencia',
                 fontsize=14, fontweight='bold', color=AZUL, y=1.01)
    plt.tight_layout()
    plt.show()


SL = dict(style={'description_width': '140px'},
          layout=Layout(width='430px'))

interact(sim1,
    s1        = FloatSlider(min=0.10, max=1.50, step=0.05, value=0.50,
                            description='s\u2081  trabajo:', **SL),
    s2        = FloatSlider(min=0.10, max=1.50, step=0.05, value=0.50,
                            description='s\u2082  consumo:', **SL),
    R0        = FloatSlider(min=2.0,  max=16.0, step=0.5,  value=8.0,
                            description='R\u2080  bliss trabajo:', **SL),
    r0        = FloatSlider(min=2.0,  max=14.0, step=0.5,  value=6.0,
                            description='r\u2080  bliss consumo:', **SL),
    b_sobre_a = FloatSlider(min=0.25, max=5.0,  step=0.25, value=1.5,
                            description='peso b/a:', **SL),
)
